# Lesson 12.7 - Distributed LLM Training Systems Playbook

## Learning Objectives
- Compare DDP, FSDP, and ZeRO trade-offs.
- Build GPU-centric training launch scaffolds.
- Estimate memory and throughput implications before expensive runs.

In [1]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch

## 1) Environment and GPU Topology Snapshot

In [2]:
gpu_count = torch.cuda.device_count()
print({"cuda": torch.cuda.is_available(), "gpu_count": gpu_count})
if gpu_count:
    for i in range(gpu_count):
        print(i, torch.cuda.get_device_name(i))

{'cuda': True, 'gpu_count': 1}
0 NVIDIA GeForce RTX 4060 Laptop GPU


## 2) Memory Planning Heuristic

In [3]:
@dataclass
class ParallelPlan:
    strategy: str
    replica_factor: float
    comm_overhead: str

plans = [
    ParallelPlan("DDP", 1.0, "medium"),
    ParallelPlan("FSDP", 0.35, "high"),
    ParallelPlan("ZeRO-3", 0.30, "high"),
    ParallelPlan("Tensor+Pipeline", 0.25, "very high"),
]

base_model_gb = 14.0
rows = []
for p in plans:
    rows.append({
        "strategy": p.strategy,
        "approx_model_memory_per_gpu_gb": round(base_model_gb * p.replica_factor, 2),
        "comm_overhead": p.comm_overhead,
    })

pd.DataFrame(rows)

,strategy,approx_model_memory_per_gpu_gb,comm_overhead
0,DDP,14.0,medium
1,FSDP,4.9,high
2,ZeRO-3,4.2,high
3,Tensor+Pipeline,3.5,very high


## 3) Distributed Launch Command Templates (GPU-centric)

In [4]:
fsdp_cmd = "torchrun --nproc_per_node=8 train.py --strategy fsdp --mixed_precision bf16"
zero_cmd = "deepspeed train.py --deepspeed ds_zero3_config.json"
tp_pp_cmd = "torchrun --nproc_per_node=8 train.py --tensor_parallel 2 --pipeline_parallel 2"

print(fsdp_cmd)
print(zero_cmd)
print(tp_pp_cmd)

torchrun --nproc_per_node=8 train.py --strategy fsdp --mixed_precision bf16
deepspeed train.py --deepspeed ds_zero3_config.json
torchrun --nproc_per_node=8 train.py --tensor_parallel 2 --pipeline_parallel 2


## 4) Training Telemetry Schema for Ops

In [5]:
telemetry = pd.DataFrame([
    {"step": 100, "tokens_per_sec": 5200, "step_time_s": 1.8, "comm_wait_pct": 19.0, "oom": 0},
    {"step": 200, "tokens_per_sec": 4980, "step_time_s": 2.0, "comm_wait_pct": 23.0, "oom": 0},
    {"step": 300, "tokens_per_sec": 4700, "step_time_s": 2.3, "comm_wait_pct": 29.0, "oom": 1},
])
telemetry

,step,tokens_per_sec,step_time_s,comm_wait_pct,oom
0,100,5200,1.8,19.0,0
1,200,4980,2.0,23.0,0
2,300,4700,2.3,29.0,1


In [6]:
alerts = []
if telemetry["comm_wait_pct"].mean() > 25:
    alerts.append("High communication overhead: review topology and sharding config.")
if telemetry["oom"].sum() > 0:
    alerts.append("OOM detected: reduce sequence length or raise sharding level.")
print(alerts if alerts else ["No critical alerts"])

['OOM detected: reduce sequence length or raise sharding level.']


## 5) Minimal PyTorch FSDP Scaffold (Reference)

In [7]:
fsdp_scaffold = """
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = build_model()
model = FSDP(model)
for batch in train_loader:
    loss = model(**batch).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
"""
print(fsdp_scaffold)


from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

model = build_model()
model = FSDP(model)
for batch in train_loader:
    loss = model(**batch).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()



## Connect to Theory
- Strategy choice should follow memory bottleneck first, then throughput profiling.
- Communication telemetry is central to scaling decisions.
- Checkpoint and failure handling must be designed before long runs.

## Case Studies & Exceptions
1. FSDP rescued memory limits but required network tuning.
2. ZeRO-3 enabled larger tuning runs with checkpoint complexity trade-offs.
3. Exception: for smaller models, DDP often remains best due to lower operational complexity.

## Interview Questions & Answers
1. **Q:** Why distributed training? **A:** Model scale exceeds single-device memory/compute.
2. **Q:** FSDP core benefit? **A:** Shards model states to reduce memory per GPU.
3. **Q:** ZeRO-3 purpose? **A:** Partition params, grads, optimizer states for maximal memory savings.
4. **Q:** Tensor vs pipeline parallelism? **A:** Intra-layer split vs inter-layer stage split.
5. **Q:** Key scaling metric? **A:** Communication wait percentage vs useful compute.
6. **Q:** Common anti-pattern? **A:** Adding GPUs without profiling interconnect bottlenecks.
7. **Q:** Why checkpoint often? **A:** Failure recovery and reproducibility.
8. **Q:** Why mixed precision? **A:** Throughput and memory efficiency gains.
9. **Q:** What is pipeline bubble? **A:** Idle stage time in pipeline schedule.
10. **Q:** One-line strategy? **A:** Use the simplest parallel setup that satisfies memory, then profile before adding complexity.